# หลักสูตรวิทยาศาสตร์ข้อมูลสำหรับผู้บริหาร: พื้นฐานและการเตรียมความพร้อม
## ตอนที่ 3: การจัดการและวิเคราะห์ตารางข้อมูลด้วย Pandas โดยเปรียบเทียบกับ Microsoft Excel
### กรณีศึกษา: ชุดข้อมูลยอดขายวิดีโอเกมทั่วโลก (Video Game Sales Dataset ผ่าน KaggleHub)

---

### วัตถุประสงค์การเรียนรู้
1. ทำความเข้าใจการเปลี่ยนผ่านจากการทำงานบน Microsoft Excel สู่การประมวลผลอัตโนมัติด้วย Pandas DataFrame
2. สามารถอ่านและเปรียบเทียบคำสั่งใน Pandas กับเครื่องมือที่คุ้นเคยใน Excel (เช่น AutoFilter, VLOOKUP, Pivot Table)
3. เข้าใจกระบวนการตรวจสอบคุณภาพข้อมูลเบื้องต้น (Data Quality Audit) ด้วยคำสั่งเชิงสถิติและโครงสร้าง บนชุดข้อมูลจริงระดับ 16,000+ แถว
4. เข้าใจตรรกะการทำความสะอาดข้อมูลสูญหาย (Missing Values) และการรวมตารางข้อมูลจากหลายแหล่ง (Data Merging)

## 1. บทนำ: ทำไมองค์กรจึงย้ายการวิเคราะห์จาก Excel มาสู่ Pandas

Microsoft Excel เป็นเครื่องมือที่ยอดเยี่ยมสำหรับการสร้างแบบจำลองการเงินและการจัดการข้อมูลทั่วไป อย่างไรก็ตาม เมื่อองค์กรขยายตัวและมีปริมาณข้อมูลเพิ่มขึ้น ข้อจำกัดของ Excel จะเริ่มส่งผลกระทบต่อประสิทธิภาพการดำเนินงาน:

| มิติการเปรียบเทียบ | Microsoft Excel | Pandas (Python) |
|---|---|---|
| **ขีดจำกัดขนาดข้อมูล** | 1,048,576 แถว (เริ่มตอบสนองช้าเมื่อเกิน 200,000 แถว) | ไม่จำกัด ขึ้นอยู่กับขนาด RAM ของระบบ (รองรับหลายล้านแถวได้อย่างรวดเร็ว) |
| **การทำงานอัตโนมัติ (Automation)** | ต้องบันทึก Macro หรือทำซ้ำด้วยมือทุกสัปดาห์/เดือน | เขียนคำสั่งครั้งเดียว สามารถรันซ้ำตามรอบเวลาได้อัตโนมัติ (Pipeline) |
| **การตรวจสอบย้อนกลับ (Auditability)** | สูตรคำนวณอาจถูกเขียนทับในบางเซลล์โดยไม่รู้ตัว (Human Error) | โค้ดถูกบันทึกเป็นขั้นตอนเชิงตรรกะ ข้อมูลต้นฉบับไม่ถูกแตะต้อง (Non-destructive) |
| **การเชื่อมต่อแหล่งข้อมูล** | การเชื่อมโยงไฟล์ข้ามเครื่องมักเกิดปัญหาลิงก์ขาด | เชื่อมต่อโดยตรงกับฐานข้อมูล SQL, Cloud Storage, และ API ได้อย่างมั่นคง |

## 2. ตารางเทียบเคียงฟังก์ชันการทำงาน: Excel เทียบกับ Pandas

| การทำงานใน Excel | เครื่องมือ / ฟังก์ชันใน Excel | คำสั่งใน Pandas |
|---|---|---|
| **เปิดไฟล์ข้อมูล** | File > Open หรือ Data Query | `pd.read_csv()` หรือ `pd.read_excel()` |
| **ดูขนาดตาราง** | ดูแถวและคอลัมน์สุดท้าย | `df.shape` (ส่งกลับจำนวนแถวและจำนวนคอลัมน์) |
| **ดูประเภทข้อมูลและช่องว่าง** | ดู Format Cells และคอลัมน์ | `df.info()` |
| **สรุปสถิติตัวเลข** | AVERAGE, STDEV, MIN, MAX | `df.describe()` |
| **การกรองข้อมูล** | ตัวกรองอัตโนมัติ (AutoFilter) | Boolean Indexing เช่น `df[df['Global_Sales'] > 10.0]` |
| **การจัดการค่าว่าง** | ค้นหา Blank Cells แล้วลบหรือเติม | `df.dropna()` หรือ `df.fillna()` |
| **การสรุปกลุ่มข้อมูล** | Pivot Table | `df.groupby('Genre')['Global_Sales'].agg(['sum', 'mean'])` |
| **การดึงข้อมูลข้ามตาราง** | VLOOKUP หรือ XLOOKUP | `pd.merge(table_a, table_b, on='Key', how='left')` |

## 3. การนำเข้าชุดข้อมูลสากลผ่าน KaggleHub และการตรวจสอบคุณภาพข้อมูล

เราจะดาวน์โหลดชุดข้อมูลจริง `gregorut/videogamesales` ผ่านไลบรารี `kagglehub` แล้วนำเข้าไฟล์ `vgsales.csv` เข้าสู่ DataFrame

คำอธิบายคอลัมน์ในชุดข้อมูล:
- `Rank`: อันดับยอดขายรวมทั่วโลก
- `Name`: ชื่อเกม
- `Platform`: แพลตฟอร์มเครื่องเล่นเกม (เช่น PS4, X360, Wii, PC)
- `Year`: ปีที่วางจำหน่าย
- `Genre`: ประเภทของเกม (เช่น Action, Sports, Shooter)
- `Publisher`: บริษัทผู้จัดจำหน่าย
- `NA_Sales`: ยอดขายในทวีปอเมริกาเหนือ (หน่วย: ล้านชุด)
- `EU_Sales`: ยอดขายในทวีปยุโรป (หน่วย: ล้านชุด)
- `JP_Sales`: ยอดขายในประเทศญี่ปุ่น (หน่วย: ล้านชุด)
- `Other_Sales`: ยอดขายในภูมิภาคอื่นๆ ทั่วโลก (หน่วย: ล้านชุด)
- `Global_Sales`: ยอดขายรวมทั่วโลก (หน่วย: ล้านชุด)

In [1]:
# ติดตั้ง kagglehub สำหรับดาวน์โหลดชุดข้อมูล
%pip install -q kagglehub

# นำเข้าไลบรารี kagglehub และ pandas
import kagglehub
import pandas as pd

# ดาวน์โหลดชุดข้อมูล Video Game Sales จาก Kaggle
path = kagglehub.dataset_download("gregorut/videogamesales")
print("ตำแหน่งไฟล์ข้อมูลในเครื่องเสมือน:", path)

# นำเข้าไฟล์ vgsales.csv เข้าสู่ตัวแปรตารางข้อมูล df (DataFrame)
df = pd.read_csv(path + "/vgsales.csv")

# แสดงผล 5 แถวแรกของตารางข้อมูลด้วย display()
display(df.head())

Note: you may need to restart the kernel to use updated packages.
ตำแหน่งไฟล์ข้อมูลในเครื่องเสมือน: C:\Users\pc\.cache\kagglehub\datasets\gregorut\videogamesales\versions\2


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


### การตรวจสอบขนาดและโครงสร้างข้อมูลด้วย `.shape` และ `.info()`
ใน Excel การเปิดไฟล์ที่มีจำนวนแถวเกินหมื่นแถวอาจเริ่มตอบสนองช้า ใน Pandas เราสามารถใช้คำสั่ง `.shape` เพื่อดูมิติข้อมูล และ `.info()` เพื่อตรวจสอบว่ามีคอลัมน์ใดที่มีค่าสูญหาย (Missing Data) หรือไม่

In [2]:
# ตรวจสอบจำนวนแถวและจำนวนคอลัมน์
print("ขนาดของตารางข้อมูล (จำนวนแถว, จำนวนคอลัมน์):", df.shape)

print("-" * 80)
# แสดงรายละเอียดของแต่ละคอลัมน์ ชนิดข้อมูล และจำนวนข้อมูลที่ไม่เป็นค่าว่าง
df.info()

ขนาดของตารางข้อมูล (จำนวนแถว, จำนวนคอลัมน์): (16598, 11)
--------------------------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  str    
 2   Platform      16598 non-null  str    
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  str    
 5   Publisher     16540 non-null  str    
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), str(4)
memory usage: 1.4 MB


### การสรุปสถิติตัวเลขภาพรวมด้วย `.describe()`
คำสั่งนี้เทียบเท่ากับการเขียนสูตรคำนวณหาค่าเฉลี่ย (Mean) ส่วนเบี่ยงเบนมาตรฐาน (Std) ค่าต่ำสุด (Min) ควอร์ไทล์ และค่าสูงสุด (Max) ของทุกคอลัมน์ตัวเลขพร้อมกันในเสี้ยววินาที

In [3]:
# สรุปสถิติเชิงพรรณนาสำหรับคอลัมน์ยอดขายและปีที่เปิดตัว
df.describe()

,Rank,Year,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
count,16598.000000,16327.000000,16598.000000,16598.000000,16598.000000,16598.000000,16598.000000
mean,8300.605254,2006.406443,0.264667,0.146652,0.077782,0.048063,0.537441
std,4791.853933,5.828981,0.816683,0.505351,0.309291,0.188588,1.555028
min,1.000000,1980.000000,0.000000,0.000000,0.000000,0.000000,0.010000
25%,4151.250000,2003.000000,0.000000,0.000000,0.000000,0.000000,0.060000
50%,8300.500000,2007.000000,0.080000,0.020000,0.000000,0.010000,0.170000
75%,12449.750000,2010.000000,0.240000,0.110000,0.040000,0.040000,0.470000
max,16600.000000,2020.000000,41.490000,29.020000,10.220000,10.570000,82.740000


## 4. การกรองข้อมูลตามเงื่อนไขทางธุรกิจ (Data Filtering: AutoFilter เทียบกับ Boolean Indexing)

ใน Excel การกรองทำได้โดยคลิกเปิด AutoFilter แล้วติ๊กเลือกค่าที่ต้องการ ซึ่งเสี่ยงต่อการที่พนักงานลืมปลดฟิลเตอร์หรือเลือกไม่ครบ
ใน Pandas จะใช้การเขียนเงื่อนไขทางตรรกะในวงเล็บก้ามปู ซึ่งมีความชัดเจน ตรวจสอบย้อนหลังได้ และรันซ้ำอัตโนมัติ

In [4]:
# ตัวอย่างที่ 1: กรองเฉพาะเกมบนแพลตฟอร์ม PlayStation 4 ('PS4')
ps4_games = df[df["Platform"] == "PS4"]
print("จำนวนเกมบนแพลตฟอร์ม PS4:", len(ps4_games))
display(ps4_games.head(3))

# ตัวอย่างที่ 2: กรองแบบหลายเงื่อนไข (เกมแนว Action ที่มียอดขายทั่วโลก Global_Sales ตั้งแต่ 5 ล้านชุดขึ้นไป)
top_action_games = df[(df["Genre"] == "Action") & (df["Global_Sales"] >= 5.0)]
print("จำนวนเกม Action ระดับยอดขายมากกว่า 5 ล้านชุด:", len(top_action_games))
display(top_action_games[["Name", "Platform", "Year", "Publisher", "Global_Sales"]].head(5))

จำนวนเกมบนแพลตฟอร์ม PS4: 336


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
33,34,Call of Duty: Black Ops 3,PS4,2015.0,Shooter,Activision,5.77,5.81,0.35,2.31,14.24
44,45,Grand Theft Auto V,PS4,2014.0,Action,Take-Two Interactive,3.80,5.81,0.36,2.02,11.98
77,78,FIFA 16,PS4,2015.0,Sports,Electronic Arts,1.11,6.06,0.06,1.26,8.49


จำนวนเกม Action ระดับยอดขายมากกว่า 5 ล้านชุด: 37


,Name,Platform,Year,Publisher,Global_Sales
16,Grand Theft Auto V,PS3,2013.0,Take-Two Interactive,21.40
17,Grand Theft Auto: San Andreas,PS2,2004.0,Take-Two Interactive,20.81
23,Grand Theft Auto V,X360,2013.0,Take-Two Interactive,16.38
24,Grand Theft Auto: Vice City,PS2,2002.0,Take-Two Interactive,16.15
38,Grand Theft Auto III,PS2,2001.0,Take-Two Interactive,13.10


## 5. การจัดการข้อมูลสูญหาย (Handling Missing Values)

เมื่อตรวจสอบด้วย `df.isna().sum()` จะพบว่าในชุดข้อมูลจริงนี้มีคอลัมน์ที่มีข้อมูลขาดหาย:
- `Year`: มีข้อมูลสูญหายประมาณ 271 รายการ
- `Publisher`: มีข้อมูลสูญหายประมาณ 58 รายการ

แนวทางจัดการทางธุรกิจ:
1. สำหรับ `Publisher` ที่ไม่ระบุ: หากเราต้องการความแม่นยำของรายงานบริษัทผู้จัดจำหน่าย เราสามารถใช้ `dropna(subset=["Publisher"])` เพื่อตัดรายการที่ไม่ระบุออก
2. สำหรับ `Year` ที่ขาดหาย: สามารถแทนที่ด้วยค่ามัธยฐาน (Median Year) หรือใช้ `fillna()` เพื่อไม่ให้สูญเสียยอดขายรวมของแถวนั้น

In [5]:
# ตรวจสอบจำนวนข้อมูลสูญหายในแต่ละคอลัมน์
print("จำนวนค่าว่างในแต่ละคอลัมน์:")
print(df.isna().sum())

print("-" * 80)

# ทำสำเนาข้อมูลเพื่อเข้าสู่กระบวนการทำความสะอาด (Non-destructive approach)
df_cleaned = df.copy()

# 1. ตัดแถวที่ไม่มีข้อมูล Publisher ออกเพื่อความถูกต้องของรายงานบริษัทคู่ค้า
df_cleaned = df_cleaned.dropna(subset=["Publisher"])

# 2. เติมเต็มค่าว่างในคอลัมน์ Year ด้วยค่ามัธยฐานของปีที่วางจำหน่าย
median_year = df_cleaned["Year"].median()
df_cleaned["Year"] = df_cleaned["Year"].fillna(median_year)

# ตรวจสอบผลลัพธ์หลังทำความสะอาด
print("จำนวนค่าว่างหลังการทำความสะอาด:")
print(df_cleaned.isna().sum())

จำนวนค่าว่างในแต่ละคอลัมน์:
Rank              0
Name              0
Platform          0
Year            271
Genre             0
Publisher        58
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64
--------------------------------------------------------------------------------
จำนวนค่าว่างหลังการทำความสะอาด:
Rank            0
Name            0
Platform        0
Year            0
Genre           0
Publisher       0
NA_Sales        0
EU_Sales        0
JP_Sales        0
Other_Sales     0
Global_Sales    0
dtype: int64


## 6. การสรุปผลเชิงกลุ่ม (GroupBy: Pivot Table ใน Excel)

คำสั่ง `.groupby()` ใน Pandas ทำงานเทียบเท่ากับการลากฟิลด์ลงใน Pivot Table ของ Excel:
- ดึงคอลัมน์หมวดหมู่ (เช่น `Genre`) มาไว้ที่ Rows
- นำคอลัมน์ตัวเลข (เช่น `Global_Sales`, `NA_Sales`, `EU_Sales`, `JP_Sales`) มาคำนวณผลรวม (Sum) หรือค่าเฉลี่ย (Mean) ในส่วน Values

In [6]:
# การทำ Pivot Table สรุปยอดขายรวมและค่าเฉลี่ย แยกตามประเภทเกม (Genre)
genre_pivot = df_cleaned.groupby("Genre").agg({
    "Global_Sales": ["sum", "mean"],
    "NA_Sales": "sum",
    "EU_Sales": "sum",
    "JP_Sales": "sum",
    "Name": "count"
})

# ปรับแต่งชื่อคอลัมน์ให้อ่านเข้าใจง่ายสำหรับรายงานฝ่ายบริหาร
genre_pivot.columns = [
    "ยอดขายรวมทั่วโลก (ล้านชุด)",
    "ยอดขายเฉลี่ยต่อเกม (ล้านชุด)",
    "ยอดขายรวมอเมริกาเหนือ (ล้านชุด)",
    "ยอดขายรวมยุโรป (ล้านชุด)",
    "ยอดขายรวมญี่ปุ่น (ล้านชุด)",
    "จำนวนเกมทั้งหมด"
]

# เรียงลำดับจากประเภทเกมที่สร้างยอดขายทั่วโลกสูงสุดลงมา
genre_pivot = genre_pivot.sort_values(by="ยอดขายรวมทั่วโลก (ล้านชุด)", ascending=False)
display(genre_pivot)

,ยอดขายรวมทั่วโลก (ล้านชุด),ยอดขายเฉลี่ยต่อเกม (ล้านชุด),ยอดขายรวมอเมริกาเหนือ (ล้านชุด),ยอดขายรวมยุโรป (ล้านชุด),ยอดขายรวมญี่ปุ่น (ล้านชุด),จำนวนเกมทั้งหมด
Genre,,,,,,
Action,1750.33,0.528960,877.54,524.92,159.49,3309
Sports,1329.35,0.567371,682.54,376.30,135.25,2343
Shooter,1037.32,0.793058,582.59,313.24,38.28,1308
Role-Playing,927.31,0.624031,327.28,188.05,352.26,1486
Platform,830.87,0.939898,447.05,201.13,130.77,884
Misc,802.15,0.468546,404.67,213.93,107.72,1712
Racing,732.03,0.586562,359.42,238.38,56.69,1248
Fighting,445.70,0.526832,221.92,100.22,87.35,846
Simulation,391.96,0.454183,183.31,113.20,63.64,863


## 7. การรวมตารางข้อมูล (Data Merging: VLOOKUP / XLOOKUP ใน Excel)

ในงานธุรกิจจริง ข้อมูลมักแยกจัดเก็บในตารางหลัก (Master Table) ตัวอย่างเช่น เรามีตารางระบุประเทศต้นกำเนิดและทวีปของบริษัทผู้จัดจำหน่าย (Publisher Headquarter Master)

ใน Excel ผู้ใช้งานต้องเขียนสูตร `=VLOOKUP(F2, Master!A:B, 2, FALSE)` ซึ่งทำงานช้าเมื่อข้อมูลมี 16,000+ แถว
ใน Pandas เราใช้คำสั่ง `pd.merge()` เพื่อผสานข้อมูลได้อย่างรวดเร็วและแม่นยำ

In [7]:
# สร้างตารางข้อมูลมาสเตอร์ของบริษัทผู้จัดจำหน่ายเกมชั้นนำ
publisher_master_data = pd.DataFrame({
    "Publisher": [
        "Nintendo", 
        "Electronic Arts", 
        "Activision", 
        "Sony Computer Entertainment", 
        "Ubisoft", 
        "Take-Two Interactive"
    ],
    "HQ_Country": ["Japan", "USA", "USA", "Japan", "France", "USA"],
    "Market_Tier": ["Tier 1", "Tier 1", "Tier 1", "Tier 1", "Tier 1", "Tier 1"]
})

print("ตารางมาสเตอร์บริษัทผู้จัดจำหน่าย:")
display(publisher_master_data)

print("-" * 80)

# ทำการ Left Merge เพื่อดึงข้อมูล HQ_Country เข้าสู่ตารางยอดขายหลัก (เทียบเท่า VLOOKUP)
df_merged = pd.merge(
    left=df_cleaned,
    right=publisher_master_data,
    on="Publisher",
    how="left"
)

# แสดงผลตัวอย่าง 5 แถวแรกที่ผ่านการจับคู่ข้อมูล
display(df_merged[["Rank", "Name", "Platform", "Publisher", "HQ_Country", "Global_Sales"]].head(5))

ตารางมาสเตอร์บริษัทผู้จัดจำหน่าย:


,Publisher,HQ_Country,Market_Tier
0,Nintendo,Japan,Tier 1
1,Electronic Arts,USA,Tier 1
2,Activision,USA,Tier 1
3,Sony Computer Entertainment,Japan,Tier 1
4,Ubisoft,France,Tier 1
5,Take-Two Interactive,USA,Tier 1


--------------------------------------------------------------------------------


,Rank,Name,Platform,Publisher,HQ_Country,Global_Sales
0,1,Wii Sports,Wii,Nintendo,Japan,82.74
1,2,Super Mario Bros.,NES,Nintendo,Japan,40.24
2,3,Mario Kart Wii,Wii,Nintendo,Japan,35.82
3,4,Wii Sports Resort,Wii,Nintendo,Japan,33.00
4,5,Pokemon Red/Pokemon Blue,GB,Nintendo,Japan,31.37


## 8. โค้ดตัวอย่างแบบบูรณาการ: สรุปผลงานของบริษัทผู้ผลิตรายใหญ่

ตัวอย่างกระบวนการประมวลผลตั้งแต่ต้นจนจบ (End-to-End Pipeline) เพื่อสรุปยอดขายรวมและส่วนแบ่งการตลาดของบริษัทระดับ Tier 1

In [8]:
# กรองเฉพาะเกมที่ผลิตโดยบริษัทระดับ Tier 1
tier1_games = df_merged[df_merged["Market_Tier"] == "Tier 1"]

# สรุปยอดขายตามประเทศสำนักงานใหญ่ (HQ_Country) และบริษัท (Publisher)
tier1_summary = tier1_games.groupby(["HQ_Country", "Publisher"]).agg({
    "Global_Sales": "sum",
    "NA_Sales": "sum",
    "JP_Sales": "sum",
    "Name": "count"
}).reset_index()

# ปรับชื่อคอลัมน์สำหรับรายงาน
tier1_summary.columns = [
    "ประเทศต้นกำเนิด", 
    "บริษัทผู้จัดจำหน่าย", 
    "ยอดขายรวมทั่วโลก (ล้านชุด)", 
    "ยอดขายอเมริกาเหนือ (ล้านชุด)", 
    "ยอดขายญี่ปุ่น (ล้านชุด)", 
    "จำนวนเกมที่เปิดตัว"
]

# จัดเรียงตามยอดขายรวมทั่วโลกจากมากไปน้อย
tier1_summary = tier1_summary.sort_values(by="ยอดขายรวมทั่วโลก (ล้านชุด)", ascending=False)
display(tier1_summary)

,ประเทศต้นกำเนิด,บริษัทผู้จัดจำหน่าย,ยอดขายรวมทั่วโลก (ล้านชุด),ยอดขายอเมริกาเหนือ (ล้านชุด),ยอดขายญี่ปุ่น (ล้านชุด),จำนวนเกมที่เปิดตัว
1,Japan,Nintendo,1786.56,816.87,455.42,703
4,USA,Electronic Arts,1110.32,595.07,14.04,1351
3,USA,Activision,727.46,429.70,6.54,975
2,Japan,Sony Computer Entertainment,607.50,265.22,74.10,683
0,France,Ubisoft,474.72,253.43,7.50,921
5,USA,Take-Two Interactive,399.54,220.49,5.83,413


## 9. ส่วนฝึกปฏิบัติสำหรับผู้เรียน (Hands-on Exercise)

### โจทย์ธุรกิจสำหรับการประเมินผล
ในฐานะผู้จัดการฝ่ายกลยุทธ์ผลิตภัณฑ์ ท่านได้รับมอบหมายให้ศึกษาตลาดเกมบนเครื่อง **PlayStation 4 (`Platform == 'PS4'`)**:
1. **โจทย์ที่ 1 (การกรองข้อมูล):** กรองข้อมูลจากตาราง `df_cleaned` โดยเลือกเฉพาะเกมบนแพลตฟอร์ม **`PS4`** ที่มียอดขายทั่วโลก **`Global_Sales >= 2.0`** (ล้านชุด) จัดเก็บในตัวแปร `top_ps4_games`
2. **โจทย์ที่ 2 (การคำนวณทางสถิติ):** จากตัวแปร `top_ps4_games` จงคำนวณ:
   - ยอดขายรวมทั่วโลกของกลุ่มนี้ (`total_global_sales`)
   - ค่าเฉลี่ยยอดขายในทวีปอเมริกาเหนือต่อเกม (`avg_na_sales`)
3. **โจทย์ที่ 3 (Pivot Table ด้วย GroupBy):** ใช้คำสั่ง `.groupby()` สรุปผลยอดขายรวมทั่วโลก (`Global_Sales`) แยกตามประเภทเกม (**`Genre`**) สำหรับกลุ่ม `top_ps4_games` พร้อมจัดเรียงจากประเภทที่ขายดีที่สุด

In [14]:
# TODO: เขียนโค้ดตอบโจทย์ที่ 1, 2, และ 3 ด้านล่างนี้
# -----------------------------------------------------------------------------
# 1. กรองเกมบนแพลตฟอร์ม PS4 ที่มียอดขาย Global_Sales >= 2.0 ล้านชุด
top_ps4_games = df[(df["Platform"] == "PS4") & (df["Global_Sales"] >= 2.0)]
print("จำนวนเกม PS4 ที่มียอดขาย >= 2.0 ล้านชุด:", len(top_ps4_games))
display(top_ps4_games.head())

# 2. คำนวณยอดขายรวมทั่วโลก และค่าเฉลี่ยยอดขายในอเมริกาเหนือ
total_global_sales = top_ps4_games["Global_Sales"].sum()
avg_na_sales = top_ps4_games["NA_Sales"].mean()

print(f"ยอดขายรวมทั่วโลกของเกม PS4 ที่มียอดขาย >= 2.0 ล้านชุด: {total_global_sales:.2f} ล้านชุด")
print(f"ค่าเฉลี่ยยอดขายในอเมริกาเหนือของเกม PS4 ที่มียอดขาย >= 2.0 ล้านชุด: {avg_na_sales:.2f} ล้านชุด")

# 3. สรุปยอดขายรวม Global_Sales แยกตาม Genre โดยใช้ groupby
genre_pivot_ps4 = top_ps4_games.groupby("Genre").agg({
    "Global_Sales": ["sum"],
})

genre_pivot_ps4.columns = [
    "ยอดขายรวมทั่วโลก (ล้านชุด)"
]

genre_pivot_ps4 = genre_pivot_ps4.sort_values(by="ยอดขายรวมทั่วโลก (ล้านชุด)", ascending=False)
display(genre_pivot_ps4)

# -----------------------------------------------------------------------------

จำนวนเกม PS4 ที่มียอดขาย >= 2.0 ล้านชุด: 39


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
33,34,Call of Duty: Black Ops 3,PS4,2015.0,Shooter,Activision,5.77,5.81,0.35,2.31,14.24
44,45,Grand Theft Auto V,PS4,2014.0,Action,Take-Two Interactive,3.80,5.81,0.36,2.02,11.98
77,78,FIFA 16,PS4,2015.0,Sports,Electronic Arts,1.11,6.06,0.06,1.26,8.49
92,93,Star Wars Battlefront (2015),PS4,2015.0,Shooter,Electronic Arts,2.93,3.29,0.22,1.23,7.67
93,94,Call of Duty: Advanced Warfare,PS4,2014.0,Shooter,Activision,2.80,3.30,0.14,1.37,7.60


ยอดขายรวมทั่วโลกของเกม PS4 ที่มียอดขาย >= 2.0 ล้านชุด: 169.44 ล้านชุด
ค่าเฉลี่ยยอดขายในอเมริกาเหนือของเกม PS4 ที่มียอดขาย >= 2.0 ล้านชุด: 1.51 ล้านชุด


,ยอดขายรวมทั่วโลก (ล้านชุด)
Genre,
Shooter,60.88
Action,52.53
Sports,34.23
Role-Playing,10.69
Racing,4.39
Misc,4.02
Fighting,2.70


---
## เฉลยและคำอธิบายแนวคิด (Solution & Managerial Concept)

ส่วนนี้จัดทำขึ้นเพื่อให้ผู้บริหารสามารถตรวจทานและทำความเข้าใจแนวทางการเขียนคำสั่งที่ถูกต้อง

In [13]:
# เฉลยโจทย์ที่ 1: การกรองข้อมูลแบบ 2 เงื่อนไขด้วยเครื่องหมาย &
top_ps4_games = df_cleaned[(df_cleaned["Platform"] == "PS4") & (df_cleaned["Global_Sales"] >= 2.0)]
print("จำนวนเกม PS4 ที่มียอดขายตั้งแต่ 2 ล้านชุดขึ้นไป:", len(top_ps4_games))
display(top_ps4_games[["Rank", "Name", "Genre", "Publisher", "Global_Sales"]].head(5))

# เฉลยโจทย์ที่ 2: การคำนวณทางสถิติ
total_global_sales = top_ps4_games["Global_Sales"].sum()
avg_na_sales = top_ps4_games["NA_Sales"].mean()
print(f"ยอดขายรวมทั่วโลกของเกมกลุ่มนี้: {total_global_sales:,.2f} ล้านชุด")
print(f"ยอดขายเฉลี่ยในอเมริกาเหนือต่อเกม: {avg_na_sales:,.2f} ล้านชุด")

# เฉลยโจทย์ที่ 3: สรุปผลด้วย groupby แยกตาม Genre พร้อมเรียงลำดับ
ps4_genre_summary = top_ps4_games.groupby("Genre")["Global_Sales"].sum().reset_index()
ps4_genre_summary.columns = ["ประเภทเกม", "ยอดขายรวมทั่วโลก (ล้านชุด)"]
ps4_genre_summary = ps4_genre_summary.sort_values(by="ยอดขายรวมทั่วโลก (ล้านชุด)", ascending=False)
display(ps4_genre_summary)

จำนวนเกม PS4 ที่มียอดขายตั้งแต่ 2 ล้านชุดขึ้นไป: 39


,Rank,Name,Genre,Publisher,Global_Sales
33,34,Call of Duty: Black Ops 3,Shooter,Activision,14.24
44,45,Grand Theft Auto V,Action,Take-Two Interactive,11.98
77,78,FIFA 16,Sports,Electronic Arts,8.49
92,93,Star Wars Battlefront (2015),Shooter,Electronic Arts,7.67
93,94,Call of Duty: Advanced Warfare,Shooter,Activision,7.60


ยอดขายรวมทั่วโลกของเกมกลุ่มนี้: 169.44 ล้านชุด
ยอดขายเฉลี่ยในอเมริกาเหนือต่อเกม: 1.51 ล้านชุด


,ประเภทเกม,ยอดขายรวมทั่วโลก (ล้านชุด)
5,Shooter,60.88
0,Action,52.53
6,Sports,34.23
4,Role-Playing,10.69
3,Racing,4.39
2,Misc,4.02
1,Fighting,2.70


### คำอธิบายแนวคิดการเลือกใช้คำสั่งสำหรับผู้บริหาร
1. **การกรองข้อมูลความจุสูง:** การใช้คำสั่งเดียวกรองข้อมูล 16,500+ แถว ช่วยประหยัดเวลาและลดความผิดพลาดของมนุษย์เมื่อเทียบกับการใช้เมาส์คลิกฟิลเตอร์ใน Excel
2. **การคำนวณที่แม่นยำ:** การดึงคอลัมน์ตัวเลขมาหา `.sum()` และ `.mean()` โดยตรง ป้องกันปัญหาช่วงเซลล์หลุด (เช่น ลืมรวมแถวที่เพิ่มเข้ามาใหม่ในสูตร `=SUM(A1:A500)`)
3. **ความพร้อมในการส่งต่อผลลัพธ์:** ตารางที่ได้จาก `.reset_index()` อยู่ในรูปแบบโครงสร้างตารางสัมพันธ์ (Relational Table) พร้อมนำไปสร้าง Dashboard หรือเชื่อมต่อกับระบบ Business Intelligence (BI) ขององค์กรได้ทันที